# 🎬 Bilibili 影片下載 + Markdown 轉換器 v2.0

完整整合版，支援 Cookie 登入，解決 Bilibili 下載限制。

**作者**：蝦仔  
**更新日期**：2025-03-05  
**版本**：v2.0 (含 Cookie 支援)

## 📋 使用說明

1. **準備 Cookie** (推薦)：
   - Chrome 安裝擴展："Get cookies.txt LOCALLY"
   - 登入 https://www.bilibili.com
   - 點擊擴展 → Export → 儲存為 `cookies.txt`

2. **設定播放清單**：修改 Step 3 嘅 URL

3. **按順序運行**每個 Step

4. **檢查輸出**：去 Google Drive 睇 Markdown 檔案

---

## Step 1: 連接 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 設定工作目錄
WORK_DIR = "/content/drive/MyDrive/bilibili-projects"
DOWNLOAD_DIR = f"{WORK_DIR}/downloads"
AUDIO_DIR = f"{WORK_DIR}/audio"
OUTPUT_DIR = f"{WORK_DIR}/output"

import os
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd {WORK_DIR}
print(f"✅ 已連接 Google Drive")
print(f"📁 工作目錄: {WORK_DIR}")

---

## Step 2: 安裝依賴

In [ ]:
print("📦 安裝依賴中...")

# 主要工具
!pip install -q yt-dlp you-get openai-whisper ffmpeg-python

# FFmpeg
!apt-get update -qq && apt-get install -y -qq ffmpeg

print("✅ 安裝完成！")

---

## Step 3: 上傳 Cookie (可選但推薦)

In [ ]:
from google.colab import files

print("📤 Cookie 上傳")
print("-" * 50)
print("說明：")
print("1. Chrome 安裝擴展：Get cookies.txt LOCALLY")
print("2. 登入 bilibili.com")
print("3. 點擊擴展 → Export → 儲存 cookies.txt")
print("4. 點擊下方按鈕上傳")
print("-" * 50)
print()

COOKIE_FILE = None

try:
    uploaded = files.upload()
    
    if 'cookies.txt' in uploaded:
        COOKIE_FILE = 'cookies.txt'
        with open(COOKIE_FILE, 'r') as f:
            lines = [l for l in f.readlines() if not l.startswith('#') and l.strip()]
        print(f"✅ Cookie 上傳成功！共 {len(lines)} 條記錄")
    else:
        print("⚠️ 無 cookies.txt，將嘗試匿名下載")
        COOKIE_FILE = None
        
except Exception as e:
    print(f"⚠️ 上傳取消或失敗: {e}")
    print("將嘗試匿名下載...")
    COOKIE_FILE = None

print(f"\n🍪 Cookie 檔案: {COOKIE_FILE if COOKIE_FILE else '無'}")

---

## Step 4: 配置設定

In [ ]:
# ==========================================
# 請修改以下設定
# ==========================================

# Bilibili 播放清單 URL
PLAYLIST_URL = "https://space.bilibili.com/1925268550/lists/839662?type=season"

# Whisper 模型: tiny, base, small, medium, large
WHISPER_MODEL = "small"  # small 係平衡選擇

# 語言: zh (中文), en (英文), ja (日文), ko (韓文)
LANGUAGE = "zh"

# 限制處理數量 (None = 全部, 數字 = 只處理前 N 條)
LIMIT = None

# ==========================================
# 設定完成
# ==========================================

print("📝 當前設定:")
print(f"  📺 播放清單: {PLAYLIST_URL}")
print(f"  🤖 Whisper 模型: {WHISPER_MODEL}")
print(f"  🌐 語言: {LANGUAGE}")
print(f"  📊 限制: {LIMIT if LIMIT else '無限制 (全部)'}")
print(f"  🍪 Cookie: {'已上傳' if COOKIE_FILE else '無 (匿名下載)'}")

---

## Step 5: 獲取播放清單資訊

In [ ]:
import subprocess
import json

def get_playlist_info(url, cookie_file=None):
    """獲取播放清單資訊"""
    cmd = [
        "yt-dlp",
        "--flat-playlist",
        "--dump-json",
        "--no-warnings"
    ]
    
    if cookie_file and os.path.exists(cookie_file):
        cmd.extend(["--cookies", cookie_file])
    
    cmd.append(url)
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    videos = []
    for line in result.stdout.strip().split('\n'):
        if line:
            try:
                data = json.loads(line)
                videos.append({
                    'id': data.get('id', ''),
                    'title': data.get('title', '未知標題'),
                    'uploader': data.get('uploader', '未知UP主'),
                    'duration': data.get('duration', 0),
                    'webpage_url': data.get('webpage_url', ''),
                    'thumbnail': data.get('thumbnail', '')
                })
            except:
                pass
    
    return videos

print("🔍 正在解析播放清單...")
print(f"URL: {PLAYLIST_URL}")
print()

videos = get_playlist_info(PLAYLIST_URL, COOKIE_FILE)

if not videos:
    print("❌ 無法獲取播放清單！請檢查:")
    print("  - URL 是否正確")
    print("  - 是否需要 Cookie")
    raise SystemExit

print(f"✅ 找到 {len(videos)} 條影片\n")

# 應用限制
if LIMIT:
    videos = videos[:LIMIT]
    print(f"📌 限制處理前 {LIMIT} 條\n")

# 顯示列表
print("📋 影片列表:")
print("-" * 60)
for i, v in enumerate(videos, 1):
    duration_min = int(v['duration'] // 60)
    print(f"{i:2d}. {v['title'][:50]} ({v['uploader']}) [{duration_min}分鐘]")

print(f"\n🎯 準備下載 {len(videos)} 條影片")

---

## Step 6: 下載影片

In [ ]:
def download_video_ytdlp(url, filename, cookie_file=None):
    """使用 yt-dlp 下載"""
    output_path = f"{DOWNLOAD_DIR}/{filename}.%(ext)s"
    
    cmd = [
        "yt-dlp",
        "--format", "best[ext=mp4]/best",
        "--output", output_path,
        "--user-agent", "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "--referer", "https://www.bilibili.com",
        "--no-warnings",
        "--progress"
    ]
    
    if cookie_file and os.path.exists(cookie_file):
        cmd.extend(["--cookies", cookie_file])
    
    cmd.append(url)
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            for f in os.listdir(DOWNLOAD_DIR):
                if f.startswith(filename):
                    return os.path.join(DOWNLOAD_DIR, f)
        
    except Exception as e:
        print(f"  yt-dlp 異常: {e}")
    
    return None


def download_video_youget(url, filename):
    """使用 you-get 備用下載"""
    try:
        cmd = [
            "you-get",
            "--output-dir", DOWNLOAD_DIR,
            "--output-filename", filename,
            url
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            for f in os.listdir(DOWNLOAD_DIR):
                if f.startswith(filename):
                    return os.path.join(DOWNLOAD_DIR, f)
    except:
        pass
    
    return None


def download_video(url, filename, cookie_file=None):
    """嘗試多種方法下載"""
    # 方法 1: yt-dlp
    path = download_video_ytdlp(url, filename, cookie_file)
    if path:
        return path, "yt-dlp"
    
    # 方法 2: you-get
    print("  🔄 yt-dlp 失敗，嘗試 you-get...")
    path = download_video_youget(url, filename)
    if path:
        return path, "you-get"
    
    return None, None


# 開始下載
print("\n" + "="*60)
print("⬇️  開始下載影片")
print("="*60 + "\n")

downloaded = []
failed = []

for i, video in enumerate(videos, 1):
    print(f"\n[{i}/{len(videos)}] {video['title']}")
    
    # 清理檔案名稱
    safe_name = "".join(c for c in video['title'] if c.isalnum() or c in " -_")[:50]
    filename = f"{i:03d}_{safe_name}"
    
    # 嘗試下載
    path, method = download_video(video['webpage_url'], filename, COOKIE_FILE)
    
    if path:
        downloaded.append({
            'video': video,
            'path': path,
            'filename': filename
        })
        print(f"  ✅ 成功 ({method}): {os.path.basename(path)}")
    else:
        failed.append(video)
        print(f"  ❌ 失敗")

# 顯示結果
print("\n" + "="*60)
print("📊 下載結果")
print("="*60)
print(f"  ✅ 成功: {len(downloaded)}/{len(videos)}")
print(f"  ❌ 失敗: {len(failed)}/{len(videos)}")

if failed:
    print("\n失敗列表:")
    for v in failed:
        print(f"  - {v['title']}")

---

## Step 7: 語音轉錄 (Whisper)

In [ ]:
import whisper

# 載入模型
print(f"🤖 載入 Whisper 模型: {WHISPER_MODEL}")
model = whisper.load_model(WHISPER_MODEL)
print("✅ 模型載入完成！\n")

In [ ]:
def extract_audio(video_path, output_name):
    """提取音軌"""
    output_path = f"{AUDIO_DIR}/{output_name}.wav"
    
    if os.path.exists(output_path):
        return output_path
    
    cmd = [
        "ffmpeg", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le",
        "-ar", "16000", "-ac", "1",
        "-y", output_path
    ]
    
    subprocess.run(cmd, capture_output=True, check=True)
    return output_path


def transcribe(audio_path):
    """轉錄音訊"""
    result = model.transcribe(audio_path, language=LANGUAGE)
    return result


def generate_markdown(video_info, transcript, output_path):
    """生成 Markdown"""
    content = f"""# {video_info['title']}

## 元數據

| 項目 | 內容 |
|------|------|
| **標題** | {video_info['title']} |
| **UP主** | {video_info['uploader']} |
| **時長** | {video_info['duration']} 秒 |
| **來源** | {video_info['webpage_url']} |

---

## 內容轉錄

"""
    
    for seg in transcript.get('segments', []):
        start_m, start_s = int(seg['start'] // 60), int(seg['start'] % 60)
        end_m, end_s = int(seg['end'] // 60), int(seg['end'] % 60)
        content += f"### {start_m:02d}:{start_s:02d} - {end_m:02d}:{end_s:02d}\n\n"
        content += f"{seg['text'].strip()}\n\n"
    
    content += "---\n\n*由 bilibili-to-markdown 自動生成*\n"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(content)
    
    return output_path


# 處理每個影片
print("\n" + "="*60)
print("📝 開始語音轉錄")
print("="*60 + "\n")

results = []

for i, item in enumerate(downloaded, 1):
    print(f"\n[{i}/{len(downloaded)}] {item['video']['title']}")
    
    # 提取音軌
    print("  🔊 提取音軌...")
    audio_path = extract_audio(item['path'], item['filename'])
    
    # 轉錄
    print("  🎯 轉錄中...")
    transcript = transcribe(audio_path)
    
    # 生成 Markdown
    md_path = f"{OUTPUT_DIR}/{item['filename']}.md"
    generate_markdown(item['video'], transcript, md_path)
    
    results.append({
        'video': item['video'],
        'markdown': os.path.basename(md_path)
    })
    
    print(f"  ✅ 完成: {os.path.basename(md_path)}")

print(f"\n📊 成功轉錄: {len(results)} 條影片")

---

## Step 8: 生成索引

In [ ]:
from datetime import datetime

summary_path = f"{OUTPUT_DIR}/summary.md"

with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("# Bilibili 播放清單轉錄索引\n\n")
    f.write(f"**生成日期**: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write(f"**總影片數**: {len(results)}\n\n")
    f.write("## 影片列表\n\n")
    f.write("| 序號 | 標題 | UP主 |\n")
    f.write("|------|------|------|\n")
    
    for i, r in enumerate(results, 1):
        title = r['video']['title']
        uploader = r['video']['uploader']
        md_file = r['markdown']
        f.write(f"| {i} | [{title}](./{md_file}) | {uploader} |\n")
    
    f.write("\n---\n\n")
    f.write("*由 bilibili-to-markdown 自動生成*\n")

print(f"✅ 索引檔案: {summary_path}")

---

## ✅ 完成！

In [ ]:
print("\n" + "="*60)
print("🎉 全部處理完成！")
print("="*60)
print(f"\n📁 檔案位置: {OUTPUT_DIR}")
print(f"\n📊 統計:")
print(f"  - 總影片: {len(videos)}")
print(f"  - 成功下載: {len(downloaded)}")
print(f"  - 成功轉錄: {len(results)}")
print(f"\n📄 輸出檔案:")
for r in results:
    print(f"  - {r['markdown']}")
print(f"\n📑 索引: summary.md")
print("\n💡 提示: 去 Google Drive 查看結果")
print(f"   {OUTPUT_DIR}")
print("="*60)